In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.datasets import load_breast_cancer

# Part 1: Regression Task (California Housing)

## Task 1: Load and Split Dataset

In [ ]:
#Load csv
df = pd.read_csv("/content/drive/MyDrive/Datasets/housing.csv")

# Handling missing values and filling
df = df.fillna(df.median(numeric_only=True))

# Converting categorical data into numerical columns
df = pd.get_dummies(df, drop_first=True)

# Seperating features and targets
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])


Training samples: 16512
Test samples: 4128


## Task2: Regressions task
Step 1: Baseline Model (No Regularization)

In [ ]:
# Baseline Linear Regression
linear_regression = LinearRegression()
linear_regression.fit(X_train, y_train)

# Predictions
y_train_pred = linear_regression.predict(X_train)
y_test_pred = linear_regression.predict(X_test)

# MSE
train_mse_linear_regression = mean_squared_error(y_train, y_train_pred)
test_mse_linear_regression = mean_squared_error(y_test, y_test_pred)

print("Baseline Linear Regression")
print("Train MSE:", train_mse_linear_regression)
print("Test MSE:", test_mse_linear_regression)

# Coefficients
print("Number of coefficients:", len(linear_regression.coef_))

Baseline Linear Regression
Train MSE: 4683203783.504253
Test MSE: 4908476721.156606
Number of coefficients: 12


Step 2: Hyperparameter Tuning Use GridSearchCV or RandomizedSearchCV to tune
hyperparameters for Ridge and Lasso regression models.

## Ridge Regression with GridSearchCV

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

ridge = Ridge()

ridge_params = {
    "alpha": [0.01, 0.1, 1, 10, 100]
}

ridge_grid = GridSearchCV(
    ridge,
    ridge_params,
    cv=5,
    scoring="neg_mean_squared_error"
)

ridge_grid.fit(X_train, y_train)

print("Best Ridge Alpha:", ridge_grid.best_params_)

best_ridge = ridge_grid.best_estimator_

# Predictions
y_train_pred_ridge = best_ridge.predict(X_train)
y_test_pred_ridge = best_ridge.predict(X_test)

print("Ridge Train MSE:", mean_squared_error(y_train, y_train_pred_ridge))
print("Ridge Test MSE:", mean_squared_error(y_test, y_test_pred_ridge))


Best Ridge Alpha: {'alpha': 1}
Ridge Train MSE: 4683383574.687479
Ridge Test MSE: 4910037869.229384


## Lasso Regression with GridSearchCV

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(max_iter=10000)

lasso_params = {
    "alpha": [0.01, 0.1, 1, 10, 100]
}

lasso_grid = GridSearchCV(
    lasso,
    lasso_params,
    cv=5,
    scoring="neg_mean_squared_error"
)

lasso_grid.fit(X_train, y_train)

print("Best Lasso Alpha:", lasso_grid.best_params_)

best_lasso = lasso_grid.best_estimator_

# Predictions
y_train_pred_lasso = best_lasso.predict(X_train)
y_test_pred_lasso = best_lasso.predict(X_test)

print("Lasso Train MSE:", mean_squared_error(y_train, y_train_pred_lasso))
print("Lasso Test MSE:", mean_squared_error(y_test, y_test_pred_lasso))


Best Lasso Alpha: {'alpha': 0.01}
Lasso Train MSE: 4683203783.920323
Lasso Test MSE: 4908478666.800429


## Step 3: Regularization Experiments (L1 vs L2)

In [ ]:
ridge_zero_coeffs = np.sum(best_ridge.coef_ == 0)
lasso_zero_coeffs = np.sum(best_lasso.coef_ == 0)

print("Ridge zero coefficients:", ridge_zero_coeffs)
print("Lasso zero coefficients:", lasso_zero_coeffs)


Ridge zero coefficients: 0
Lasso zero coefficients: 0


Lasso did not produce zero coefficients because the optimal regularization strength was small. This suggests that all features were important. Using a larger alpha forced sparsity but reduced accuracy, showing the bias–variance tradeoff.

#  Task2: Classification Task (Diabetes)

## Step 1: Baseline Model Logistic Regression

In [ ]:
X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
log_reg = LogisticRegression(max_iter=10000)
log_reg.fit(X_train, y_train)

# Predictions
y_train_pred = log_reg.predict(X_train)
y_test_pred = log_reg.predict(X_test)

print("Baseline Logistic Regression")
print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

Baseline Logistic Regression
Train Accuracy: 0.9582417582417583
Test Accuracy: 0.956140350877193


Step2: Hyper Parameter tuning

In [ ]:
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"]
}

log_reg = LogisticRegression(
    solver="liblinear",
    max_iter=10000
)

grid = GridSearchCV(
    log_reg,
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

best_log_reg = grid.best_estimator_

# Accuracy
print("Train Accuracy:", accuracy_score(y_train, best_log_reg.predict(X_train)))
print("Test Accuracy:", accuracy_score(y_test, best_log_reg.predict(X_test)))


Best Parameters: {'C': 100, 'penalty': 'l1'}
Train Accuracy: 0.989010989010989
Test Accuracy: 0.9824561403508771


Regularization Experiments (L1 vs L2)

In [ ]:
# L1 Model
l1_model = LogisticRegression(
    C=grid.best_params_["C"],
    penalty="l1",
    solver="liblinear",
    max_iter=10000
)

l1_model.fit(X_train, y_train)

# L2 Model
l2_model = LogisticRegression(
    C=grid.best_params_["C"],
    penalty="l2",
    solver="liblinear",
    max_iter=10000
)

l2_model.fit(X_train, y_train)

# Accuracy comparison
print("L1 Train Accuracy:", accuracy_score(y_train, l1_model.predict(X_train)))
print("L1 Test Accuracy:", accuracy_score(y_test, l1_model.predict(X_test)))

print("L2 Train Accuracy:", accuracy_score(y_train, l2_model.predict(X_train)))
print("L2 Test Accuracy:", accuracy_score(y_test, l2_model.predict(X_test)))

# Sparsity
print("L1 zero coefficients:", np.sum(l1_model.coef_ == 0))
print("L2 zero coefficients:", np.sum(l2_model.coef_ == 0))


L1 Train Accuracy: 0.989010989010989
L1 Test Accuracy: 0.9824561403508771
L2 Train Accuracy: 0.9692307692307692
L2 Test Accuracy: 0.956140350877193
L1 zero coefficients: 9
L2 zero coefficients: 0


L1 regularization achieved higher training and test accuracy than L2 and produced a sparse model by setting 9 coefficients to zero. This shows that L1 can perform feature selection while maintaining good performance. L2 reduced coefficient sizes without removing features, resulting in slightly lower accuracy. Overall, regularization helps control overfitting, but too much regularization can reduce model performance.